# INFORM — Selectivity optimization example

Lightweight example showing how to optimize a selective stimulation protocol
and visualize the result, using the `selectivity_optimization` package.

Heavy batch runs (all populations) are done with `scripts/run_selectivity.py`.
This notebook demonstrates a single population and the plotting helpers.


## Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# --- Parameters (edit these) ---
PROJECT_FOLDER = Path(r"C:/Users/laura/OneDrive/Desktop/Projects/Models")
NERVE_FOLDER   = "nerve_1"     # circular nerve; for the median use its block folder
TRIAL          = "cross"
POPULATION_ID  = 1
ACTIVE_SITES   = 3
MAX_STIM       = 1.0

# --- Legacy pickle compatibility (same shim as the run scripts) ---
import nerve_model.experiment as _experiment
import nerve_model.fiber_population as _fiber_population
import nerve_model.nerve_section as _nerve_section
import nerve_model.histological_nerve_section as _histological_nerve_section
import nerve_model.recruitment_curves as _recruitment_curves
import nerve_model.implant as _implant

sys.modules["experiment"] = _experiment
sys.modules["fiber_population"] = _fiber_population
sys.modules["nerve_section"] = _nerve_section
sys.modules["recruitment_curves"] = _recruitment_curves
sys.modules["implant"] = _implant
sys.modules["nerve_section_2"] = _histological_nerve_section

from selectivity_optimization import (
    params_to_selectivity_rasp,
    selectivity_eval,
    plot_matrix,
    plot_color_section,
    off_diagonal_frobenius_norm,
)
print("Imports OK")

## Load one population and its localization result

In [ ]:
import pickle

def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

exp_base  = PROJECT_FOLDER / "Median nerve" / "experiments" / NERVE_FOLDER / TRIAL
surr_base = PROJECT_FOLDER / "Median nerve" / "surrogate_experiments" / NERVE_FOLDER / TRIAL

# population (true) experiment
pop_file = exp_base / f"experiment_pop_{POPULATION_ID}trial_{TRIAL}.pkl"   # circular naming
true_experiment = load_pickle(pop_file)

# localization result carries the inferred (pred) experiment
loc_file = surr_base / "results" / f"results_{TRIAL}_pop_{POPULATION_ID}_with_topography.pkl"
loc = load_pickle(loc_file)
pred_experiment = loc["pred_experiment"] if isinstance(loc, dict) else loc[-3]

n_clusters = true_experiment.fiber_population.n_groups
print("Clusters:", n_clusters, "| sites:", true_experiment.implant.n_sites)

## Optimize selectivity per cluster (PSO)
Optimize on the inferred nerve using the margin metric (`selectivity_opt`).

In [ ]:
from pyswarms.single import GlobalBestPSO

n_sites = pred_experiment.implant.n_sites
options = {"c1": 1.5, "c2": 1.5, "w": 0.9}
bounds = (np.repeat(np.array([[-MAX_STIM],[0]]), n_sites),
          np.repeat(np.array([[MAX_STIM],[1]]), n_sites))

protocols, recruitments = [], []
for cluster_idx in range(n_clusters):
    optimizer = GlobalBestPSO(n_particles=20, dimensions=2*n_sites,
                              options=options, bounds=bounds,
                              ftol_iter=50, ftol=1e-10)
    _, best_pos = optimizer.optimize(
        params_to_selectivity_rasp, iters=10,
        n_sites=n_sites, n_active_sites=ACTIVE_SITES,
        true_population=pred_experiment.fiber_population,
        experiment=pred_experiment, musc_selective=cluster_idx,
        batch_size=20,
    )
    sp, rc, _ = params_to_selectivity_rasp(
        best_pos, n_sites, ACTIVE_SITES,
        pred_experiment.fiber_population, pred_experiment,
        musc_selective=cluster_idx, batch_size=1, return_recruitment=True,
    )
    protocols.append(sp)
    recruitments.append(rc)

rc_pred_array = np.array(recruitments).squeeze()
print("Optimization done.")

## Cross-evaluation on the true nerve
Apply the inferred-optimized protocols to the true nerve, summarize with `selectivity_eval`.

In [ ]:
rc_true_array = np.zeros((n_clusters, n_clusters))
for cluster_idx in range(n_clusters):
    rc_true_array[cluster_idx, :] = true_experiment.compute_recruitment_patterns(
        stimulation_protocols=protocols[cluster_idx], method="from_self"
    )

sel_crossed = selectivity_eval(rc_true_array, normalize=True, squared=True)
_, fro_ratio_pred = off_diagonal_frobenius_norm(rc_pred_array)
_, fro_ratio_true = off_diagonal_frobenius_norm(rc_true_array)
print("Crossed selectivity:", np.round(sel_crossed, 3))
print("Off-diag ratio  pred:", round(fro_ratio_pred,3), "| true:", round(fro_ratio_true,3))

## Visualize

In [ ]:
fig = plot_matrix(rc_pred_array, size=n_clusters,
                  title=f"Inferred  (mean sel {np.mean(sel_crossed):.3f})")
plt.show()

fig = plot_matrix(rc_true_array, size=n_clusters, title="True (cross-evaluated)")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
plot_color_section(
    experiment=true_experiment,
    fiber_in_fascicle=true_experiment.fiber_population.cluster_ids.astype(int),
    ax=ax,
)
ax.set_title("True section")
plt.show()